In [ ]:
pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 48.7 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/ChemFoundationModels/ChemLLMBench.git

Cloning into 'ChemLLMBench'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 244 (delta 78), reused 96 (delta 43), pack-reused 87 (from 1)
Receiving objects: 100% (244/244), 4.27 MiB | 9.96 MiB/s, done.
Resolving deltas: 100% (103/103), done.


In [ ]:
import openai
import random
import pandas as pd
from tqdm import tqdm
import numpy as np
from sklearn.metrics import f1_score,accuracy_score
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import DataStructs
from rdkit.Chem import rdMolDescriptors
from rdkit import Chem
import warnings
from rdkit import RDLogger
# from steamship import Steamship
import datetime
import os
import time

In [ ]:
random.seed(42)
#read bace dataset
bace = pd.read_csv("/content/ChemLLMBench/data/property_prediction/BACE.csv")
sample_size = 80
bace_sample= bace.sample(sample_size)
bace.drop(bace_sample.index, inplace = True)

In [ ]:
mkdir /content/results/

In [ ]:
bace_sample.to_csv("/content/results/BACE_test.csv",index = False)
bace.to_csv("/content/results/BACE_train.csv",index =False)
print(bace_sample['Class'].value_counts())

Class
0    45
1    35
Name: count, dtype: int64


In [ ]:
def generate_response_by_gpt35(prompt, model_engine="gpt-4o-mini"):
  client = openai.OpenAI(api_key="YOUR_API_KEY_HERE")
  completion = client.chat.completions.create(
      model=model_engine,
      temperature=1,
      max_tokens=100,
      n=1,
      messages=[
          {"role": "user", "content": prompt}
          ]
    )

    # Extract messages from all generated choices
  messages = [choice.message.content.strip() for choice in completion.choices]
  return messages

In [ ]:
# random sampling
def radom_sample_examples(bace,sample_size):
    positive_examples = bace[bace["Class"] == 1].sample(int(sample_size/2))
    negative_examples = bace[bace["Class"] == 0].sample(int(sample_size/2))
    smiles = positive_examples["mol"].tolist() + negative_examples["mol"].tolist()

    class_label = positive_examples["Class"].tolist() + negative_examples["Class"].tolist()
    #convert 1 to "Yes" and 0 to "No"" in class_label
    class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bace_examples = list(zip(smiles, class_label))
    return bace_examples

In [ ]:
# scaffold sampling
def top_k_scaffold_similar_molecules(target_smiles, bace_data, k):
    #drop the target_smiles from the dataset
    bace_data = bace_data[bace_data["mol"] != target_smiles]
    molecule_smiles_list = bace_data['mol'].tolist()
    label_list = bace_data['Class'].tolist()
    label_list = ["Yes" if i == 1 else "No" for i in label_list]

    target_mol = Chem.MolFromSmiles(target_smiles)
    if target_mol is not None:
        target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    else:
        print("Error: Unable to create a molecule from the provided SMILES string.")
        #drop the target_smiles from the dataset
        return None

    target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    target_fp = rdMolDescriptors.GetMorganFingerprint(target_scaffold, 2)
    RDLogger.DisableLog('rdApp.warning')
    warnings.filterwarnings("ignore", category=UserWarning)
    similarities = []

    for i,smiles in enumerate(molecule_smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        try:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaffold_fp = rdMolDescriptors.GetMorganFingerprint(scaffold, 2)
            tanimoto_similarity = DataStructs.TanimotoSimilarity(target_fp, scaffold_fp)
            # print(tanimoto_similarity)
            similarities.append((smiles, tanimoto_similarity,label_list[i]))
        except:
            continue
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_5_similar_molecules = similarities[:k]
    return top_5_similar_molecules

In [ ]:
sample_size = 2
target_smiles = "O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1"
random_examples = radom_sample_examples(bace_sample,sample_size)
print("randomly sampling examples", radom_sample_examples(bace_sample,sample_size))
print("scaffold sampling examples", top_k_scaffold_similar_molecules(target_smiles, bace_sample,sample_size))

randomly sampling examples [('FC1(F)COC(=NC1(C)c1cc(NC(=O)c2nn(cc2)C(F)F)ccc1F)N', 'Yes'), ('Clc1cc(cc(Cl)c1NC(=O)C)C\\N=C(\\NC(=O)Cn1c2c(cccc2)cc1)/N', 'No')]
scaffold sampling examples [('Fc1ccc(OC)cc1-c1cc2c(Oc3c(cc(OC)cc3)C23N=C(OC3)N)cc1', 0.6941176470588235, 'No'), ('FC(F)(F)Oc1ccc(cc1)-c1cc2c(Oc3c(cc(OC)cc3)C23N=C(OC3)N)cc1', 0.6941176470588235, 'No')]


[21:07:46] DEPRECATION WARNING: please use MorganGenerator


In [ ]:
def create_bace_prompt(input_smiles,pp_examples):
    prompt = "You are an expert chemist, your task is to predict the property of molecule using your experienced chemical property prediction knowledge.\nPlease strictly follow the format, no other information can be provided. Given the SMILES string of a molecule, predict the molecular properties of a given chemical compound based on its structure, by analyzing wether it can inhibit(Yes) the Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1) or cannot inhibit(No) BACE1. Consider factors such as molecular weight, atom count, bond types, and functional groups in order to assess the compound's drug-likeness and its potential to serve as an effective therapeutic agent for Alzheimer's disease,please answer with only Yes or No. A few examples are provided in the beginning.\n"
    for example in pp_examples:
        prompt += f"SMILES: {example[0]}\nBACE-1 Inhibit: {example[-1]}\n"
    prompt += f"SMILES: {input_smiles}\nBACE-1 Inhibit:\n"
    return prompt

In [ ]:
input_smiles = "O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1"
example_prompt = create_bace_prompt(input_smiles,random_examples)
print(example_prompt)

You are an expert chemist, your task is to predict the property of molecule using your experienced chemical property prediction knowledge.
Please strictly follow the format, no other information can be provided. Given the SMILES string of a molecule, predict the molecular properties of a given chemical compound based on its structure, by analyzing wether it can inhibit(Yes) the Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1) or cannot inhibit(No) BACE1. Consider factors such as molecular weight, atom count, bond types, and functional groups in order to assess the compound's drug-likeness and its potential to serve as an effective therapeutic agent for Alzheimer's disease,please answer with only Yes or No. A few examples are provided in the beginning.
SMILES: FC1(F)COC(=NC1(C)c1cc(NC(=O)c2ncc(cc2)C#C)ccc1F)N
BACE-1 Inhibit: Yes
SMILES: Brc1ccccc1C1C[NH2+]CC1C(=O)N1CCOCC1c1ccccc1
BACE-1 Inhibit: No
SMILES: O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1
BACE-1 Inhibit

In [ ]:
import time

In [ ]:
model = 'gpt-o4'  # keep as provided; ensure your backend routes this to a valid model
sample_num = 7
detail_save_folder = '/content/results/'  # path to save the generated result
paras = 0
sample_method = ['random','scaffold']

for sample_method in sample_method:
  detail_predict_file = detail_save_folder + 'test_{}_{}_{}_{}.csv'.format('bace', model, sample_num, sample_method)
  log_file = detail_save_folder + 'test_{}_{}_{}_{}.log'.format('bace', model, sample_num, sample_method)
  print(detail_predict_file)
  print()

  if os.path.exists(detail_predict_file):
    detail_results = pd.read_csv(detail_predict_file).values.tolist()
  else:
    detail_results = []

  now = datetime.datetime.now()
  date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
  with open(log_file, "a") as file:
    file.write("=" * 30 + date_time_str + "=" * 30 + "\n")

  # We now generate exactly ONE prediction per example to respect quotas
  out_cols = ['bace_smiles', 'class_label', 'pred']

/content/results/test_bace_gpt-o4_7_random.csv

/content/results/test_bace_gpt-o4_7_scaffold.csv



In [ ]:
# random sampling
detail_predict_file = '/content/results/test_bace_gpt-o4_4_random.csv'
detail_results = []   # <-- this must be a list, not the file path

para_index = 0
bace_examples = radom_sample_examples(bace, sample_num)

for i in tqdm(range(0, len(bace_sample))):
    if para_index < 0:
        para_index += 1
        continue

    example = [(bace_sample.iloc[i]['mol'], bace_sample.iloc[i]['Class'])]
    pred_y = []
    generated_results = []

    for text in example:
        prompt = create_bace_prompt(text[0], bace_examples)
        with open(log_file, "a") as file:
            file.write(prompt + "\n")
            file.write("=" * 50 + "\n")

        pred_text = generate_response_by_gpt35(prompt)
        time.sleep(21)  # or smaller + exponential backoff if rate limits occur

        generated_results.append(pred_text)
        detail_results.append([text[0]] + [text[-1]] + pred_text)

        if (i + 1) % 10 == 0:
            details_df = pd.DataFrame(
                detail_results,
                columns=['bace_smiles', 'class_label', 'pred']
            )
            details_df.to_csv(detail_predict_file, index=False)
            print('save file')

# after loop ends, save final version
details_df = pd.DataFrame(
    detail_results,
    columns=['bace_smiles', 'class_label', 'pred']
)
details_df.to_csv(detail_predict_file, index=False)


In [ ]:
# scaffold sampling
detail_predict_file = '/content/results/test_bace_gpt-o4_4_scaffold.csv'
para_index = 0
for i in tqdm(range(0, len(bace_sample))):
  if para_index < 0:
    para_index += 1
    continue

  example = [(bace_sample.iloc[i]['mol'], bace_sample.iloc[i]['Class'])]

  for text in example:
    bace_examples = top_k_scaffold_similar_molecules(text[0], bace, sample_num)
    prompt = create_bace_prompt(text[0], bace_examples)
    with open(log_file, "a") as file:
      file.write(prompt + "\n")
      file.write("=" * 50 + "\n")

    pred_text = generate_response_by_gpt35(prompt)
    time.sleep(21)  # or smaller + exponential backoff if rate limits occur

    generated_results.append(pred_text)
    detail_results.append([text[0]] + [text[-1]] + pred_text)

    if (i + 1) % 10 == 0:
      details_df = pd.DataFrame(
        detail_results,
        columns=['bace_smiles', 'class_label', 'pred']
      )
      details_df.to_csv(detail_predict_file, index=False)
      print('save file')

# after loop ends, save final version
details_df = pd.DataFrame(
    detail_results,
    columns=['bace_smiles', 'class_label', 'pred']
)
details_df.to_csv(detail_predict_file, index=False)

In [ ]:
def create_bace_prompt_zero_shot(input_text,pp_examples):
    prompt = """You are an expert chemist, your task is to predict the property of molecule using your experienced chemical property
    prediction knowledge.\nPlease strictly follow the format, no other information can be provided. Given the SMILES string of a
    molecule, predict the molecular properties of a given chemical compound based on its structure, by analyzing wether it can
    inhibit(Yes) the Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1) or cannot inhibit(No) BACE1. Consider factors
    such as molecular weight, atom count, bond types, and functional groups in order to assess the compound's drug-likeness and its
    potential to serve as an effective therapeutic agent for Alzheimer's disease,please answer with only Yes or No. A template is
    provided in the beginning.\n"""
    for example in pp_examples:
        prompt += f"SMILES: {example[0]}\nBACE-1 Inhibit: {example[-1]}\n"
    prompt += f"SMILES: {input_text}\nBACE-1 Inhibit:\n"
    return prompt

In [ ]:
label = []
accs = []
f1_scores_hiv = []
epochs = 5
performance_results = []
detail_save_folder = '/content/results/zero-shot/'
few_shot_examples = (["SMILES1","Yes"],["SMILES2","No"])
paras = 0

In [ ]:
if paras < 0:
  paras += 1

detail_predict_file = detail_save_folder + 'zero_shot_{}_{}.csv'.format('bace', model)
log_file = detail_save_folder + 'zero_shot_{}_{}.log'.format('bace', model)
print(detail_predict_file)
print()

# Create the directory if it doesn't exist
os.makedirs(detail_save_folder, exist_ok=True)

if os.path.exists(detail_predict_file):
  detail_results = pd.read_csv(detail_predict_file)
  #convert the column to list
  detail_results = detail_results.values.tolist()
else:
  detail_results = []

# append new date
# Get the current date and time
now = datetime.datetime.now()
# Convert the date and time to a string
date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
with open(log_file, "a") as file:
  file.write("=" * 30 + date_time_str + "=" * 30 + "\n")
para_index = 0

/content/results/zero-shot/zero_shot_bace_gpt-o4.csv



In [ ]:

for i in tqdm(range(0, len(bace_sample))):
  # print(para_index)
  if para_index < 0:
    para_index += 1
    continue
  example = [(bace_sample.iloc[i]['mol'],bace_sample.iloc[i]['Class'])]
  pred_y = []
  generated_results = []
  for text in example:
    prompt = create_bace_prompt_zero_shot(text[0],few_shot_examples)
    # print(prompt)
    with open(log_file, "a") as file:
      file.write(prompt + "\n")
      file.write("=" * 50 + "\n")

    pred_text = generate_response_by_gpt35(prompt)
    time.sleep(30)  # or smaller + exponential backoff if rate limits occur

    generated_results.append(pred_text)
    detail_results.append([text[0]] + [text[-1]] + pred_text)

    if (i + 1) % 10 == 0:
      details_df = pd.DataFrame(
        detail_results,
        columns=['bace_smiles', 'class_label', 'pred']
      )
      details_df.to_csv(detail_predict_file, index=False)
      print('save file')

  # after loop ends, save final version
details_df = pd.DataFrame(
  detail_results,
  columns=['bace_smiles', 'class_label', 'pred']
  )
details_df.to_csv(detail_predict_file, index=False)

In [ ]:
import time
import openai
from tqdm import tqdm

def safe_generate(prompt, max_retries=10):
    for attempt in range(max_retries):
        try:
            return generate_response_by_gpt35(prompt)  # your function
        except openai.RateLimitError as e:
            # Parse recommended wait time if available; else backoff
            wait_time = 25 + attempt * 10  # e.g. 25, 35, 45...
            print(f"Rate limit hit. Sleeping for {wait_time}s. Attempt {attempt+1}/{max_retries}")
            time.sleep(wait_time)
        except Exception as e:
            # Other errors: maybe raise or handle separately
            raise e
    raise RuntimeError("Max retries exceeded")

detail_results = []

for i in tqdm(range(0, len(bace_sample))):
    if para_index < 0:
        para_index += 1
        continue

    example = [(bace_sample.iloc[i]['mol'], bace_sample.iloc[i]['Class'])]

    for text in example:
        prompt = create_bace_prompt_zero_shot(text[0], few_shot_examples)

        with open(log_file, "a") as file:
            file.write(prompt + "\n")
            file.write("=" * 50 + "\n")

        pred_text = safe_generate(prompt)  # <-- use retry wrapper here
        time.sleep(25)  # extra padding between calls, optional

        generated_results.append(pred_text)
        detail_results.append([text[0]] + [text[-1]] + pred_text)

        if (i + 1) % 10 == 0:
            details_df = pd.DataFrame(
                detail_results,
                columns=['bace_smiles', 'class_label', 'pred']
            )
            details_df.to_csv(detail_predict_file, index=False)
            print('save file')

# final save
details_df = pd.DataFrame(
    detail_results,
    columns=['bace_smiles', 'class_label', 'pred']
)
details_df.to_csv(detail_predict_file, index=False)


In [ ]:
!zip -r /content/results.zip /content/results/

You can download the zipped file `results.zip` from the file browser.